In [1]:
import os

from dotenv import load_dotenv

load_dotenv()
import uuid

import pandas as pd
from ddgs import DDGS
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
print("\n" + "=" * 70)
print(" CREATING MULTI-SOURCE DATA")
print("=" * 70)

# Source 1: Company Documents (Text)
print("\n Creating company documents...")
company_docs = [
    Document(
        page_content="TechCorp Q4 2024 Financial Report: Revenue reached $10.5M, up 25% YoY. Net profit: $2.1M. Key growth drivers: AI products (+40%) and cloud services (+30%).",
        metadata={
            "source": "company_financials.pdf",
            "type": "document",
            "date": "2024-12-31",
            "department": "Finance",
        },
    ),
    Document(
        page_content="TechCorp Remote Work Policy 2025: Employees may work remotely up to 4 days/week. Core hours: 10 AM - 3 PM EST. All remote workers must attend weekly team meetings.",
        metadata={
            "source": "hr_policies.pdf",
            "type": "document",
            "date": "2025-01-15",
            "department": "HR",
        },
    ),
    Document(
        page_content="Project Phoenix Status: Phase 2 completed on schedule. Budget: $500K (95% utilized). Team: 12 members. Next milestone: Phase 3 launch in Q2 2025.",
        metadata={
            "source": "project_status.docx",
            "type": "document",
            "date": "2025-01-20",
            "department": "Engineering",
        },
    ),
]
print(f"    Created {len(company_docs)} company documents")

# Source 2: Product Catalog (Structured Data)
print("\n Creating product catalog...")
products_data = {
    "Product": [
        "AI Assistant Pro",
        "Cloud Storage Plus",
        "Analytics Dashboard",
        "Security Suite",
    ],
    "Price": ["$99/month", "$49/month", "$199/month", "$149/month"],
    "Category": ["AI/ML", "Storage", "Analytics", "Security"],
    "Rating": [4.8, 4.5, 4.7, 4.9],
    "Users": ["50K+", "100K+", "25K+", "40K+"],
}
products_df = pd.DataFrame(products_data)

# Convert structured data to documents
product_docs = []
for idx, row in products_df.iterrows():
    content = f"{row['Product']}: ${row['Price']} - {row['Category']} category. Rating: {row['Rating']}/5.0 with {row['Users']} active users."
    product_docs.append(
        Document(
            page_content=content,
            metadata={
                "source": "product_catalog",
                "type": "structured_data",
                "category": row["Category"],
            },
        )
    )
print(f"    Created {len(product_docs)} product entries")

# Source 3: Customer Reviews (Text)
print("\n Creating customer reviews...")
review_docs = [
    Document(
        page_content="Review by John D.: AI Assistant Pro is amazing! Increased our productivity by 40%. Worth every penny. 5 stars!",
        metadata={
            "source": "customer_reviews",
            "type": "review",
            "rating": 5,
            "product": "AI Assistant Pro",
        },
    ),
    Document(
        page_content="Review by Sarah M.: Cloud Storage Plus is reliable and fast. Great value for money. Seamless integration with our tools. 4 stars.",
        metadata={
            "source": "customer_reviews",
            "type": "review",
            "rating": 4,
            "product": "Cloud Storage Plus",
        },
    ),
    Document(
        page_content="Review by Mike R.: Analytics Dashboard provides incredible insights. The visualizations are top-notch. Highly recommended! 5 stars.",
        metadata={
            "source": "customer_reviews",
            "type": "review",
            "rating": 5,
            "product": "Analytics Dashboard",
        },
    ),
]
print(f"   Created {len(review_docs)} customer reviews")

# Source 4: Knowledge Base Articles
print("\n Creating knowledge base articles...")
kb_docs = [
    Document(
        page_content="How to integrate AI Assistant Pro: Use our REST API with your API key. Documentation: docs.techcorp.com/api. Support available 24/7.",
        metadata={
            "source": "knowledge_base",
            "type": "kb_article",
            "topic": "integration",
        },
    ),
    Document(
        page_content="Security best practices: Enable 2FA, use strong passwords, rotate API keys quarterly, monitor access logs, and enable audit trails.",
        metadata={
            "source": "knowledge_base",
            "type": "kb_article",
            "topic": "security",
        },
    ),
]
print(f"    Created {len(kb_docs)} knowledge base articles")

# Combine all documents
all_docs = company_docs + product_docs + review_docs + kb_docs
print(f"\n Total documents from all sources: {len(all_docs)}")
print("=" * 70)


 CREATING MULTI-SOURCE DATA

 Creating company documents...
    Created 3 company documents

 Creating product catalog...
    Created 4 product entries

 Creating customer reviews...
   Created 3 customer reviews

 Creating knowledge base articles...
    Created 2 knowledge base articles

 Total documents from all sources: 12


In [3]:
print("\n" + "=" * 70)
print(" PROCESSING MULTI-SOURCE DATA")
print("=" * 70)

print("\n Chunking documents...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(all_docs)
print(f"    Created {len(splits)} chunks")

print("\n Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("   Embedding model loaded")

print("\n Creating vector database...")
vectorstore = Chroma.from_documents(
    documents=splits, embedding=embeddings, collection_name="multi_source_rag"
)
print("    Vector database created")

print("\n Setting up retriever...")
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


@tool(response_format="content_and_artifact")
def search_docs(query: str):
    """Search the knowledge base for passages relevant to the query."""
    docs = retriever.invoke(query)
    content = "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)
    return content, docs  # content goes to the LLM, docs go to the artifact


print("    Retriever configured")

print("\n All sources indexed and ready!")
print("=" * 70)


 PROCESSING MULTI-SOURCE DATA

 Chunking documents...
    Created 12 chunks

 Loading embedding model...


W1027 12:21:20.255000 37098 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

   Embedding model loaded

 Creating vector database...
    Vector database created

 Setting up retriever...
    Retriever configured

 All sources indexed and ready!


In [ ]:
print("\n Setting up web search...")


def _format_hits(hits: list[dict], k: int = 5) -> str:
    lines = []
    for i, h in enumerate(hits[:k], 1):
        title = h.get("title") or h.get("heading") or ""
        url = h.get("href") or h.get("url") or ""
        snippet = h.get("body") or h.get("description") or h.get("snippet") or ""
        lines.append(f"{i}. {title}\n   {url}\n   {snippet}")
    return "\n".join(lines) if lines else "No results."


search = DDGS()  # Initialize DuckDuckGo Search


@tool
def web_search(query: str, max_results: int = 5, time_range: str | None = None) -> str:
    """DuckDuckGo web search. Returns up to `max_results` compact results with URLs and snippets.
    time_range can be 'd','w','m','y' (day, week, month, year) for recency, or None."""
    try:
        with DDGS() as ddg:
            hits = list(ddg.text(query, max_results=max_results, timelimit=time_range))
        return _format_hits(hits, k=max_results)
    except Exception as e:
        return f"Search error: {e}"


 Setting up web search...


In [3]:
print("\n" + "=" * 70)
print(" INITIALIZING AI COMPONENTS")
print("=" * 70)

print("\n Initializing GPT-5 Nano...")
llm = ChatOpenAI(
    model=os.getenv("GPT_MODEL", "your-gpt-model"),
    base_url=os.getenv("PORTKEY_BASE_URL", "your-base-url"),
    api_key=os.getenv("PORTKEY_API_KEY", "your-api-key"),
)
print(llm.invoke("Say hi in five words.").content)


 INITIALIZING AI COMPONENTS

 Initializing GPT-5 Nano...
Hi there, hope you're well.


In [6]:
print("\n Setting up agent...")

SYSTEM_PROMPT = """
You are a helpful AI assistant with access to multiple data sources:
- Internal documents (company info, products, reviews, knowledge base)
- Web search (for current/latest information)

Instructions:
- Use the provided context to answer comprehensively
- Cite your sources (e.g., "According to the financial report..." or "Web search shows...")
- If information is from web search, mention it's current/latest information
- If you don't have enough information, say so clearly
"""

agent = create_agent(
    model=llm,
    tools=[search_docs, web_search],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)


def new_config():
    return {"configurable": {"thread_id": str(uuid.uuid4())}}


config = new_config()


 Setting up agent...


In [8]:
def _text(content) -> str:
    """Normalise message content (str or list of content blocks) to plain text."""
    if isinstance(content, str):
        return content
    return "".join(
        b.get("text", "") if isinstance(b, dict) else str(b) for b in content
    )


def analyze_sources(source_docs):
    """Group retrieved documents by type -> set of source names."""
    sources_by_type = {}
    for doc in source_docs:
        source_type = doc.metadata.get("type", "unknown")
        source_name = doc.metadata.get("source", "unknown")
        sources_by_type.setdefault(source_type, set()).add(source_name)
    return sources_by_type


def chat(question, config=None, show_sources=True):
    config = config or new_config()
    print(f"\n You: {question}")

    msgs = agent.invoke({"messages": [HumanMessage(question)]}, config)["messages"]
    answer = _text(msgs[-1].content)
    print(f"\n Assistant: {answer}")

    # Only look at messages from the current turn (thread history is included in msgs)
    last_human = max(i for i, m in enumerate(msgs) if m.type == "human")
    turn = msgs[last_human:]

    tools_called = [tc["name"] for m in turn if m.type == "ai" for tc in m.tool_calls]
    docs = [
        d for m in turn if isinstance(m, ToolMessage) and m.artifact for d in m.artifact
    ]

    if show_sources:
        print(f"\n Tools called: {tools_called or 'none'}")
        if docs:
            for t, names in analyze_sources(docs).items():
                print(f" Internal sources [{t}]: {sorted(names)}")
        elif "search_docs" in tools_called:
            print(" Internal sources: none relevant")
    print("\n" + "-" * 60)
    return {"answer": answer, "source_documents": docs, "tools_called": tools_called}

In [9]:
# ============================================================================
# STEP 9: DEMO QUERIES
# ============================================================================
print("\n" + "=" * 70)
print(" MULTI-SOURCE RAG DEMONSTRATION")
print("=" * 70)
print("\nRunning demo queries that combine multiple data sources...\n")

# Query 1: Financial + Product data
chat(
    question="What was our Q4 2024 revenue and what are our top-rated products?",
    config=config,
    show_sources=True,
)

# Query 2: Product + Reviews
chat(
    question="Tell me about the AI Assistant Pro - pricing, rating, and what customers say about it.",
    config=config,
    show_sources=True,
)

# Query 3: Policy + Knowledge Base
chat(
    question="What's our remote work policy and what security practices should remote workers follow?",
    config=config,
    show_sources=True,
)

# Query 4: Cross-source analysis
chat(
    question="Which products have the best customer ratings and how do they contribute to our revenue?",
    config=config,
    show_sources=True,
)

# Query 5: Project + Product
chat(
    question="What's the status of Project Phoenix and which products could benefit from it?",
    config=config,
    show_sources=True,
)


 MULTI-SOURCE RAG DEMONSTRATION

Running demo queries that combine multiple data sources...


 You: What was our Q4 2024 revenue and what are our top-rated products?

 Assistant: Here’s what I found from our internal sources:

- Q4 2024 revenue
  - Revenue: $10.5 million
  - Year-over-year growth: +25%
  - Net profit: $2.1 million
  - Key growth drivers: AI products (+40%), cloud services (+30%)
  - Source: TechCorp Q4 2024 Financial Report (company_financials.pdf)

- Top-rated products (highest ratings, per our product catalog)
  - Security Suite — 4.9/5.0 rating; $149/month; 40k+ active users
  - AI Assistant Pro — 4.8/5.0 rating; $99/month; 50k+ active users
  - Analytics Dashboard — 4.7/5.0 rating; $199/month; 25k+ active users
  - Source: Product Catalog

If you’d like, I can also show updated monthly revenue by product or export this as a quick summary report.

 Tools called: ['search_docs', 'search_docs']
 Internal sources [document]: ['company_financials.pdf', 'project_status.

{'answer': 'Here’s the latest as reflected in our internal project docs:\n\nProject Phoenix status\n- Phase progress: Phase 2 has been completed on schedule.\n- Next milestone: Phase 3 launch is targeted for Q2 2025.\n- Budget and resources: $500K total budget; about 95% utilized so far.\n- Team: 12 members working on the project.\n\nPotential products that could benefit (based on typical outcomes of this kind of platform/project)\n- AI Assistant Pro: If Phoenix includes AI integration capabilities or improved data pipelines, AI features could become faster, more reliable, and easier to deploy across customer deployments.\n- Analytics Dashboard: Improvements in data ingestion, processing performance, or new analytics capabilities would directly elevate this product, given its data-centric nature.\n- Security Suite: Any enhancements around policy enforcement, monitoring, or secure access could strengthen security offerings across all products.\n- Cloud Services (if applicable): Platform

In [ ]:
# # ============================================================================
# # STEP 10: INTERACTIVE MODE
# # ============================================================================
# print("\n" + "=" * 70)
# print(" INTERACTIVE MODE")
# print("=" * 70)
# print("\nNow you can ask your own questions!")
# print("The system will intelligently search across all sources.")
# print("\nCommands:")
# print("  'sources' - Show all available sources")
# print("  'stats'   - Show source statistics")
# print("  'clear'   - Clear conversation memory")
# print("  'quit'    - Exit")
# print("\n" + "-" * 70 + "\n")

# while True:
#     try:
#         user_input = input(" You: ").strip()

#         if not user_input:
#             continue

#         if user_input.lower() in ["quit", "exit", "q"]:
#             print("\n Thanks for trying Multi-Source RAG!")
#             break

#         if user_input.lower() == "sources":
#             print("\n Available Sources:")
#             print("-" * 70)
#             print(
#                 " Documents: company_financials.pdf, hr_policies.pdf, project_status.docx"
#             )
#             print(" Structured Data: product_catalog")
#             print(" Reviews: customer_reviews")
#             print(" Knowledge Base: kb_article")
#             if web_search_enabled:
#                 print(" Web Search: Enabled")
#             print("-" * 70 + "\n")
#             continue

#         if user_input.lower() == "stats":
#             print("\n Data Statistics:")
#             print("-" * 70)
#             print(f"Total documents: {len(all_docs)}")
#             print(f"Total chunks: {len(splits)}")
#             print(f"Company docs: {len(company_docs)}")
#             print(f"Product entries: {len(product_docs)}")
#             print(f"Customer reviews: {len(review_docs)}")
#             print(f"KB articles: {len(kb_docs)}")
#             print("-" * 70 + "\n")
#             continue

#         if user_input.lower() == "clear":
#             memory.clear()
#             print("🧹 Memory cleared!\n")
#             continue

#         # Process question
#         chat(user_input)

#     except KeyboardInterrupt:
#         print("\n\n Chat interrupted. Goodbye!")
#         break
#     except Exception as e:
#         print(f"\n Error: {e!s}")
#         print("Please try again.\n")